In [1]:
#Library importations
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

In [2]:
#Directory of the dataset
directory = "C:/Users/j-a-b/Documents/OpenClassrooms/Projet 5 MH/Data/"
#Pattern to match the CSV files
pattern = os.path.join(directory, "*.csv")
#List to hold the dataframes
file_list = glob.glob(pattern)
#Read each CSV file and append it into a unified dataframe
pmvl = pd.concat((pd.read_csv(file, sep=",") for file in file_list), ignore_index=True)

In [3]:
#Add PFMTitres for more info about the positions
pfm = pd.read_csv("C:\\Users\\j-a-b\\Documents\\OpenClassrooms\\Projet 5 MH\\Data\\dataset_model\\PFM_PFMtitres_cleaned.csv", sep=";")

In [4]:
#Join the two dataframes on the "ISIN" and PMVL[ISIN] column
pmvl = pd.merge(pmvl, pfm, left_on="PMVL[ISIN]", right_on="ISIN", how="left")

In [6]:
#Create a new column target
#Have We asserted the PMVL[PMVL Estimé] on the next day?
#yes=True, no=False
#We take the today's PMVL[PMVL Estimé] we compare it with the PMVL[PRMP PMVL] of tomorrow
#threshold +- 5%
#Keys to group by:
keys = [
    "PMVL[ENTITE]", 
    "PMVL[Selected Fund code]", 
    "PMVL[Ref Unik Asset]",
    "PMVL[ISIN]"
]

# 1. Sort so that inside each key-combination, dates are in order
pmvl_sorted = pmvl.sort_values(
    by=keys + ["PMVL[Holding date]"]
)

# 2. Compute "tomorrow's PRMP PMVL" per (ENTITE, Fund, Asset, ISIN)
pmvl_sorted["PRMP_PMVL_tomorrow"] = (
    pmvl_sorted
    .groupby(keys)["PMVL[PRMP PMVL]"]
    .shift(-1)
)

# 3. Compare today's estimate with tomorrow's realized PRMP PMVL (±5%)
pmvl_sorted["Target"] = (
    (pmvl_sorted["PRMP_PMVL_tomorrow"] >= pmvl_sorted["PMVL[PMVL Estimé]"] * 0.95) &
    (pmvl_sorted["PRMP_PMVL_tomorrow"] <= pmvl_sorted["PMVL[PMVL Estimé]"] * 1.05)
).astype(int)

# 4. Drop duplicates per key + day (if needed)
pmvl_sorted = pmvl_sorted.drop_duplicates(
    subset=keys + ["PMVL[Holding date]"]
)

# 5. Reorder columns
pmvl_sorted = pmvl_sorted[
    keys 
    + ["PMVL[Holding date]", "PMVL[PRMP PMVL]", "PMVL[PMVL Estimé]"] + ["PRMP_PMVL_tomorrow", "Target"]
    + [c for c in pmvl_sorted.columns if c not in keys + ["PMVL[Holding date]", "Target", "PMVL[PRMP PMVL]", "PMVL[PMVL Estimé]", "PRMP_PMVL_tomorrow"]]
]

#drop line where PRMP_PMVL_tomorrow is null, because next day that position doesn't exist anymore (was closed)
pmvl_sorted = pmvl_sorted[~pmvl_sorted["PRMP_PMVL_tomorrow"].isnull()]

#csv exportation
#pmvl_sorted.to_csv("C:/Users/j-a-b/Documents/OpenClassrooms/Projet 5 MH/Notebook/pmvl_sorted.csv", index=False)

In [7]:
#Assign a ticker "pas de ticker" to all the lines, because we don't have this information in the dataset, but it's required for the model, we will use the "PMVL[Selected Fund code]" as a proxy for the ticker, because it's the only column that can be used as a unique identifier for each line, and it contains a lot of missing values, but we will fill them with "pas de ticker" to avoid losing too much data.
pmvl_sorted["PMVL[Parametres_Indices.TICKER]"] = pmvl_sorted["PMVL[Parametres_Indices.TICKER]"].fillna("pas de ticker")
#Filling missing values in "PMVL[Ref Unik Codes]" with "pas de code", because it's the only column that can be used as a unique identifier for each line, and it contains a lot of missing values, but we will fill them with "pas de code" to avoid losing too much data.
pmvl_sorted["PMVL[Ref Unik Codes]"] = pmvl_sorted["PMVL[Ref Unik Codes]"].fillna("pas de code")

In [8]:
import numpy as np
import pandas as pd

def add_features(X, numerical_features):
    X = X.copy()

    # Ensure holding date is datetime
    X["PMVL[Holding date]"] = pd.to_datetime(X["PMVL[Holding date]"])

    # Clean numeric columns
    for col in numerical_features:
        X[col] = (
            X[col]
            .astype(str)
            .str.replace(" ", "", regex=False)
            .replace("nan", np.nan)
            .astype(float)
        )
        X[col] = X[col].fillna(X[col].median())

    # Avoid division by zero
    eps = 1e-9
    denom_est = X["PRMP_PMVL_tomorrow"].replace(0, eps)
    denom_real = X["PMVL[PRMP PMVL]"].replace(0, eps)

    # Ratios
    X["Ratio_Estimate"] = X["PMVL[PMVL Estimé]"] / denom_est
    X["Ratio_Realized"] = X["PRMP_PMVL_tomorrow"] / denom_real

    # Percentage error
    X["Percentage_Error"] = (X["PRMP_PMVL_tomorrow"] - X["PMVL[PMVL Estimé]"]) / denom_est * 100

    # Calendar features
    X["Holding_Month"] = X["PMVL[Holding date]"].dt.month
    X["Holding_Day"] = X["PMVL[Holding date]"].dt.day

    return X

fe_columns = ["Ratio_Estimate", "Ratio_Realized", "Percentage_Error",
              "Holding_Month", "Holding_Day"]

In [9]:
pmvl_sorted["Target"] .value_counts()

Target
1    8481
0    2909
Name: count, dtype: int64

In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

# 0) Start from pmvl_sorted with ticker/code already filled
pmvl_sorted["PMVL[Holding date]"] = pd.to_datetime(pmvl_sorted["PMVL[Holding date]"])
pmvl_sorted = pmvl_sorted.sort_values(
    by=[
        "PMVL[ENTITE]",
        "PMVL[Selected Fund code]",
        "PMVL[Ref Unik Asset]",
        "PMVL[ISIN]",
        "PMVL[Holding date]",
    ]
)

target_col = "Target"
y = pmvl_sorted[target_col]

categorical_features = [
    "PMVL[ENTITE]", 
    "PMVL[Selected Fund code]", 
    "PMVL[Ref Unik Asset]",
    "PMVL[Parametres_Indices.TICKER]",
    "PMVL[CIC]",
    "PMVL[3A]",
    "PMVL[Ptf name]",
    "Instrument type (Decalog)",
    "Issr country",
    "Rating S2",
]

numerical_features = [
    "PMVL[PRMP PMVL]", 
    "PMVL[PMVL Estimé]", 
    "PMVL[PFMIndice.Perf J / J-1]",
    "PMVL[PRMP VNC]",
    "PMVL[Quantity]",
    "PMVL[Quote]",
    "PMVL[PRMP MtM]",
    ]

# 1) Build base X and apply feature engineering
X_raw = pmvl_sorted[
    numerical_features
    + categorical_features
    + ["PMVL[Holding date]", "PRMP_PMVL_tomorrow"]
]

X_fe = add_features(X_raw, numerical_features=numerical_features)

# final numeric = original numerics + engineered numerics
numeric_all = numerical_features + fe_columns
categorical_all = categorical_features

# 2) ColumnTransformer for RF
preprocess_rf = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_all),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
          categorical_all),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess_rf),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=42,
            class_weight="balanced_subsample",
        )),
    ]
)

# 3) Time-series cross-validation
tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_fe), start=1):
    X_train, X_val = X_fe.iloc[train_idx], X_fe.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf_pipeline.fit(X_train, y_train)
    y_val_proba = rf_pipeline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_val_proba)
    print(f"Fold {fold} ROC-AUC: {auc:.4f}")

2026/04/08 22:16:58 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '1718295a95404f05b1927dfbd9cfdeb8', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/04/08 22:16:59 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <http

Fold 1 ROC-AUC: 1.0000


2026/04/08 22:17:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/08 22:17:20 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types

Fold 2 ROC-AUC: 0.9999


2026/04/08 22:17:37 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/08 22:17:40 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types

Fold 3 ROC-AUC: 1.0000


2026/04/08 22:17:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/08 22:17:59 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types

Fold 4 ROC-AUC: 1.0000


2026/04/08 22:18:15 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/08 22:18:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types

Fold 5 ROC-AUC: 0.9962


In [30]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score
)
import numpy as np

# Chronological 80/20 split by index (data already sorted by time)
split_point = int(len(X_fe) * 0.8)
X_train, X_test = X_fe.iloc[:split_point], X_fe.iloc[split_point:]
y_train, y_test = y.iloc[:split_point], y.iloc[split_point:]

rf_pipeline.fit(X_train, y_train)

# Predictions
y_train_pred = rf_pipeline.predict(X_train)
y_test_pred = rf_pipeline.predict(X_test)
y_train_proba = rf_pipeline.predict_proba(X_train)[:, 1]
y_test_proba = rf_pipeline.predict_proba(X_test)[:, 1]

# ROC-AUC
roc_train = roc_auc_score(y_train, y_train_proba)
roc_test = roc_auc_score(y_test, y_test_proba)

print(f"ROC-AUC train: {roc_train:.4f}")
print(f"ROC-AUC test : {roc_test:.4f}")

# Classification reports (precision, recall, F1 per class)
print("=== Train classification report ===")
print(classification_report(y_train, y_train_pred, digits=3))

print("=== Test classification report ===")
print(classification_report(y_test, y_test_pred, digits=3))

# Confusion matrices
print("=== Train confusion matrix ===")
print(confusion_matrix(y_train, y_train_pred))

print("=== Test confusion matrix ===")
print(confusion_matrix(y_test, y_test_pred))

# Precision-Recall AUC (very relevant for imbalanced data)
ap_train = average_precision_score(y_train, y_train_proba)
ap_test = average_precision_score(y_test, y_test_proba)

print(f"Average precision (PR-AUC) train: {ap_train:.4f}")
print(f"Average precision (PR-AUC) test : {ap_test:.4f}")

ROC-AUC train: 1.0000
ROC-AUC test : 0.9977
=== Train classification report ===
              precision    recall  f1-score   support

           0      0.998     1.000     0.999      2334
           1      1.000     0.999     1.000      6778

    accuracy                          1.000      9112
   macro avg      0.999     1.000     0.999      9112
weighted avg      1.000     1.000     1.000      9112

=== Test classification report ===
              precision    recall  f1-score   support

           0      0.998     0.859     0.923       575
           1      0.955     0.999     0.976      1703

    accuracy                          0.964      2278
   macro avg      0.976     0.929     0.950      2278
weighted avg      0.966     0.964     0.963      2278

=== Train confusion matrix ===
[[2334    0]
 [   4 6774]]
=== Test confusion matrix ===
[[ 494   81]
 [   1 1702]]
Average precision (PR-AUC) train: 1.0000
Average precision (PR-AUC) test : 0.9992


In [38]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("pmvl_random_forest_v2")  # new experiment name

mlflow.sklearn.autolog(log_models=True)  # tracks params, metrics, model, etc.

In [39]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn import set_config
import matplotlib.pyplot as plt
import pandas as pd
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name="rf_with_instr_features"):
    # Fit on the current train slice
    rf_pipeline.fit(X_train, y_train)

    # Predictions and probabilities on the SAME slices
    y_train_pred = rf_pipeline.predict(X_train)
    y_test_pred = rf_pipeline.predict(X_test)
    y_train_proba = rf_pipeline.predict_proba(X_train)[:, 1]
    y_test_proba = rf_pipeline.predict_proba(X_test)[:, 1]

    # --- train metrics ---
    acc_train = accuracy_score(y_train, y_train_pred)
    f1_train = f1_score(y_train, y_train_pred, pos_label=1)
    rec_train = recall_score(y_train, y_train_pred, pos_label=1)
    roc_train = roc_auc_score(y_train, y_train_proba)
    ap_train = average_precision_score(y_train, y_train_proba)

    # --- test metrics ---
    acc_test = accuracy_score(y_test, y_test_pred)
    f1_test = f1_score(y_test, y_test_pred, pos_label=1)
    rec_test = recall_score(y_test, y_test_pred, pos_label=1)
    roc_test = roc_auc_score(y_test, y_test_proba)
    ap_test = average_precision_score(y_test, y_test_proba)

    mlflow.log_metrics({
        "acc_train": acc_train,
        "acc_test": acc_test,
        "f1_train": f1_train,
        "f1_test": f1_test,
        "recall_train": rec_train,
        "recall_test": rec_test,
        "roc_auc_train": roc_train,
        "roc_auc_test": roc_test,
        "pr_auc_train": ap_train,
        "pr_auc_test": ap_test,
    })

    # 1) ROC curve (test)
    RocCurveDisplay.from_predictions(y_test, y_test_proba)
    plt.title("ROC curve - test")
    plt.savefig("roc_curve_test.png", bbox_inches="tight")
    plt.close()
    mlflow.log_artifact("roc_curve_test.png")

    # 2) Precision-Recall curve (test)
    PrecisionRecallDisplay.from_predictions(y_test, y_test_proba)
    plt.title("Precision-Recall curve - test")
    plt.savefig("pr_curve_test.png", bbox_inches="tight")
    plt.close()
    mlflow.log_artifact("pr_curve_test.png")

    # 3) Feature importance bar chart (top 20)
    set_config(transform_output="pandas")
    # IMPORTANT: TRANSFORM, not fit_transform -> keep same fitted preprocessor
    X_encoded = rf_pipeline.named_steps["preprocess"].transform(X_train)
    rf = rf_pipeline.named_steps["model"]
    importances = rf.feature_importances_

    feat_imp = pd.Series(importances, index=X_encoded.columns).sort_values(ascending=False)
    top20 = feat_imp.head(20)

    plt.figure(figsize=(8, 6))
    top20[::-1].plot(kind="barh")
    plt.title("Top 20 feature importances")
    plt.tight_layout()
    plt.savefig("feature_importances_top20.png", bbox_inches="tight")
    plt.close()
    mlflow.log_artifact("feature_importances_top20.png")

    # 4) Classification report as CSV
    from sklearn.metrics import classification_report
    report = classification_report(y_test, y_test_pred, output_dict=True)
    report_df = pd.DataFrame(report).T
    report_df.to_csv("classification_report_test.csv")
    mlflow.log_artifact("classification_report_test.csv")

2026/04/08 22:44:58 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/08 22:45:02 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\j-a-b\anaconda3\Lib\site-packages\mlflow\types

In [37]:
import mlflow
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: sqlite:///C:/Users/j-a-b/Documents/OpenClassrooms/Projet%205%20MH/Notebook/mlflow.db
